In [11]:
#imports

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

# where it is being run
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(device)

cpu


In [12]:
# import the preprocessed dataset 
path = "/kaggle/input/datasets/nghinguyenn/mediapipe-9000"

X_list = []
y_list = []

# ONLY 0–8 because y_chunk_9 doesn't exist
for i in range(9):
    X_list.append(np.load(f"{path}/X_chunk_{i}.npy"))
    y_list.append(np.load(f"{path}/y_chunk_{i}.npy"))

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)

print(X.shape)
print(y.shape)

(9000, 37, 63)
(9000,)


In [13]:
# normalize data

X = X[:, :, :63]

X = (
    X - X.mean(axis=1, keepdims=True)
) / (
    X.std(axis=1, keepdims=True) + 1e-6
)

In [14]:
# convert to tensors

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [15]:
# train/validation split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [16]:
# data loaders

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=64
)

In [17]:
# transformer model 

class GestureTransformer(nn.Module):
    def __init__(self, input_dim=63, model_dim=128, num_heads=4,
                 num_layers=3, num_classes=27):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, model_dim)

        self.pos_embedding = nn.Parameter(torch.zeros(37, model_dim))
        nn.init.trunc_normal_(self.pos_embedding, std=0.02)
    
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=num_heads,
            dim_feedforward=256,
            dropout=0.2,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.dropout = nn.Dropout(0.1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(model_dim),
            nn.Linear(model_dim, num_classes)
        )

    def forward(self, x):

        x = self.input_proj(x)

        x = x + self.pos_embedding

        x = self.transformer(x)

        x = self.dropout(x)

        x = x[:, -1] + x.mean(dim=1)

        return self.classifier(x)

In [18]:
# initialize model
model = GestureTransformer(
    input_dim=63,
    num_classes=27
).to(device)

# optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=3
)

In [19]:
# loss & optimizer

import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight

# force y into clean numpy 1D ints to compute classes safely
y = np.array(y)
y = y.astype(int)
y = y.reshape(-1)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y),
    y=y
)

weights = torch.tensor(weights, dtype=torch.float32).to(device)

criterion = torch.nn.CrossEntropyLoss(weight=weights)

In [20]:
# training loop

best_val = 0

patience = 5
bad_epochs = 0

y = y.astype(np.int64)

for epoch in range(20):

    model.train()

    train_correct = 0
    train_total = 0
    train_loss = 0

    for xb, yb in train_loader:

        yb = yb.long()
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        out = model(xb)

        loss = criterion(out, yb)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)

        preds = out.argmax(dim=1)

        train_correct += (preds == yb).sum().item()
        train_total += yb.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # validation
    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            out = model(xb)

            preds = out.argmax(dim=1)

            val_correct += (preds == yb).sum().item()
            val_total += yb.size(0)

    val_acc = val_correct / val_total

    scheduler.step(val_acc)
    
    print(
        f"Epoch {epoch} | "
        f"Loss: {train_loss:.3f} | "
        f"Train Acc: {train_acc:.3f} | "
        f"Val Acc: {val_acc:.3f}"
    )

    if val_acc > best_val:
    
        best_val = val_acc
        bad_epochs = 0
    
        torch.save(
            model.state_dict(),
            "best_model.pth"
        )
    
        print("Saved best model!")
    
    else:
        bad_epochs += 1
    
    if bad_epochs >= patience:
        print("Early stopping triggered!")
        break

Epoch 0 | Loss: 2.704 | Train Acc: 0.212 | Val Acc: 0.382
Saved best model!
Epoch 1 | Loss: 1.747 | Train Acc: 0.454 | Val Acc: 0.526
Saved best model!
Epoch 2 | Loss: 1.466 | Train Acc: 0.537 | Val Acc: 0.546
Saved best model!
Epoch 3 | Loss: 1.319 | Train Acc: 0.583 | Val Acc: 0.597
Saved best model!
Epoch 4 | Loss: 1.216 | Train Acc: 0.626 | Val Acc: 0.634
Saved best model!
Epoch 5 | Loss: 1.124 | Train Acc: 0.653 | Val Acc: 0.625
Epoch 6 | Loss: 1.083 | Train Acc: 0.659 | Val Acc: 0.657
Saved best model!
Epoch 7 | Loss: 1.025 | Train Acc: 0.683 | Val Acc: 0.660
Saved best model!
Epoch 8 | Loss: 0.957 | Train Acc: 0.710 | Val Acc: 0.671
Saved best model!
Epoch 9 | Loss: 0.952 | Train Acc: 0.705 | Val Acc: 0.654
Epoch 10 | Loss: 0.933 | Train Acc: 0.708 | Val Acc: 0.657
Epoch 11 | Loss: 0.918 | Train Acc: 0.715 | Val Acc: 0.662
Epoch 12 | Loss: 0.904 | Train Acc: 0.723 | Val Acc: 0.668
Epoch 13 | Loss: 0.766 | Train Acc: 0.763 | Val Acc: 0.703
Saved best model!
Epoch 14 | Loss: 0.726